In [3]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns


TX = daily maximum temperature
TX = daily minimum temperature

In [72]:
import yaml
 
try:
    with open("../config.yaml", "r") as file:
        config = yaml.safe_load(file)
except:
    print("Yaml configuration file not found!")

In [17]:
tx=pd.read_csv(config['data']['clean']['file10'], sep=';')
tn=pd.read_csv(config['data']['clean']['file9'], sep=';')

In [18]:
tx.shape

(17155, 2)

In [19]:
tn.shape

(17155, 2)

In [20]:
display(tx.head())
display(tn.head())

,date,tx
0,1979-01-01,2.3
1,1979-01-02,1.6
2,1979-01-03,1.3
3,1979-01-04,-0.3
4,1979-01-05,5.6


,date,tn
0,1979-01-01,-7.5
1,1979-01-02,-7.5
2,1979-01-03,-7.2
3,1979-01-04,-6.5
4,1979-01-05,-1.4


In [21]:
tx.describe()

,tx
count,17155.000000
mean,15.733702
std,6.489315
min,-3.300000
25%,10.900000
50%,15.400000
75%,20.600000
max,40.200000


In [22]:
tn.describe()

,tn
count,17155.000000
mean,7.495482
std,5.340470
min,-11.800000
25%,3.458333
50%,7.600000
75%,11.775000
max,21.200000


In [23]:
tx.isna().sum()

date    0
tx      0
dtype: int64

In [24]:
tn.isna().sum()

date    0
tn      0
dtype: int64

In [25]:
tx.columns = tx.columns.str.strip()
tn.columns = tn.columns.str.strip()

In [26]:
temp=pd.merge(tx[["date", "tx"]],tn[["date", "tn"]],on="date",how="inner")

# 1. CLIMPACT MADNESS

In [92]:
rr=pd.read_csv('../data/raw/RR.csv', sep=';')

In [93]:
rr.columns = rr.columns.str.strip()

In [94]:
rr.Q_RR.value_counts()

Q_RR
0    17153
9       14
Name: count, dtype: int64

In [96]:
heathrow=pd.merge(temp,rr[["DATE", "RR", "Q_RR"]],on="DATE",how="inner")

In [97]:
heathrow.shape

(17167, 7)

In [98]:
heathrow.head()

,DATE,TX,Q_TX,TN,Q_TN,RR,Q_RR
0,19790101,23,0,-75,0,4,0
1,19790102,16,0,-75,0,0,0
2,19790103,13,0,-72,0,0,0
3,19790104,-3,0,-65,0,0,0
4,19790105,56,0,-14,0,0,0


In [99]:
heathrow=heathrow.drop(columns=["Q_TX", "Q_TN"])

In [100]:
heathrow.Q_RR.value_counts()

Q_RR
0    17153
9       14
Name: count, dtype: int64

In [101]:
heathrow.isna().sum()

DATE    0
TX      0
TN      0
RR      0
Q_RR    0
dtype: int64

In [102]:
heathrow=heathrow.drop(columns=["Q_RR"])

In [103]:
heathrow["DATE"] = pd.to_datetime(heathrow["DATE"].astype(str), format="%Y%m%d")
heathrow["TX"] = heathrow["TX"] / 10.0   
heathrow["TN"] = heathrow["TN"] / 10.0

In [104]:
heathrow.dtypes

DATE    datetime64[us]
TX             float64
TN             float64
RR               int64
dtype: object

In [105]:
# converting rain precipitaion to mm and using the name required by climpact:
heathrow['PR']=heathrow["RR"] / 10

In [106]:
heathrow.head()

,DATE,TX,TN,RR,PR
0,1979-01-01,2.3,-7.5,4,0.4
1,1979-01-02,1.6,-7.5,0,0.0
2,1979-01-03,1.3,-7.2,0,0.0
3,1979-01-04,-0.3,-6.5,0,0.0
4,1979-01-05,5.6,-1.4,0,0.0


In [107]:
heathrow=heathrow.drop(columns=["RR"])

In [108]:
heathrow.head()

,DATE,TX,TN,PR
0,1979-01-01,2.3,-7.5,0.4
1,1979-01-02,1.6,-7.5,0.0
2,1979-01-03,1.3,-7.2,0.0
3,1979-01-04,-0.3,-6.5,0.0
4,1979-01-05,5.6,-1.4,0.0


In [109]:
heathrow["year"]= heathrow["DATE"].dt.year
heathrow["month"]= heathrow["DATE"].dt.month
heathrow["day"]= heathrow["DATE"].dt.day   
heathrow.head()

,DATE,TX,TN,PR,year,month,day
0,1979-01-01,2.3,-7.5,0.4,1979,1,1
1,1979-01-02,1.6,-7.5,0.0,1979,1,2
2,1979-01-03,1.3,-7.2,0.0,1979,1,3
3,1979-01-04,-0.3,-6.5,0.0,1979,1,4
4,1979-01-05,5.6,-1.4,0.0,1979,1,5


In [110]:
heathrow=heathrow.drop(columns=["DATE"])
heathrow.head()

,TX,TN,PR,year,month,day
0,2.3,-7.5,0.4,1979,1,1
1,1.6,-7.5,0.0,1979,1,2
2,1.3,-7.2,0.0,1979,1,3
3,-0.3,-6.5,0.0,1979,1,4
4,5.6,-1.4,0.0,1979,1,5


In [111]:
heathrow.dtypes

TX       float64
TN       float64
PR       float64
year       int32
month      int32
day        int32
dtype: object

In [112]:
heathrow = heathrow.rename(columns={"year": "Year","month": "Month","day": "Day"})
heathrow = heathrow[["Year", "Month", "Day", "PR", "TN", "TX"]]
heathrow.head()

,Year,Month,Day,PR,TN,TX
0,1979,1,1,0.4,-7.5,2.3
1,1979,1,2,0.0,-7.5,1.6
2,1979,1,3,0.0,-7.2,1.3
3,1979,1,4,0.0,-6.5,-0.3
4,1979,1,5,0.0,-1.4,5.6


In [113]:
# Filtering dates:
# heathrow = heathrow[(heathrow["Year"] >= 1991) & (heathrow["Year"] <= 2020)].copy()

In [114]:
heathrow.shape

(17167, 6)

In [115]:
heathrow = heathrow.replace(-999.9, -99.9)

In [116]:
heathrow.to_csv("/Users/iirene/Desktop/heathrow.txt",index=False, header=False, sep="\t")

# 2. BASELINE CALCULATION

In [27]:
temp.head()

,date,tx,tn
0,1979-01-01,2.3,-7.5
1,1979-01-02,1.6,-7.5
2,1979-01-03,1.3,-7.2
3,1979-01-04,-0.3,-6.5
4,1979-01-05,5.6,-1.4


In [29]:
temp.dtypes

date        str
tx      float64
tn      float64
dtype: object

In [33]:
# converto date time and degrees celsius:
temp["date"] = pd.to_datetime(temp["date"])
#temp["TX"] = temp["TX"] / 10.0   
#temp["TN"] = temp["TN"] / 10.0

In [34]:
temp=temp.sort_values("date").reset_index(drop=True)

In [35]:
temp.head()

,date,tx,tn
0,1979-01-01,2.3,-7.5
1,1979-01-02,1.6,-7.5
2,1979-01-03,1.3,-7.2
3,1979-01-04,-0.3,-6.5
4,1979-01-05,5.6,-1.4


In [36]:
temp["year"]= temp["date"].dt.year
temp["month"]= temp["date"].dt.month
temp["day"]= temp["date"].dt.day  

In [37]:
#Dropping the 29th of february:

temp = temp.drop(temp[(temp["month"] == 2) & (temp["day"] == 29)].index)

In [38]:
temp["doy"]   = temp["date"].dt.dayofyear

print(temp.head())
print(f"\nTotal rows: {len(temp)}  |  Date range: {temp['date'].min().date()} → {temp['date'].max().date()}")

        date   tx   tn  year  month  day  doy
0 1979-01-01  2.3 -7.5  1979      1    1    1
1 1979-01-02  1.6 -7.5  1979      1    2    2
2 1979-01-03  1.3 -7.2  1979      1    3    3
3 1979-01-04 -0.3 -6.5  1979      1    4    4
4 1979-01-05  5.6 -1.4  1979      1    5    5

Total rows: 17155  |  Date range: 1979-01-01 → 2025-12-31


In [40]:
print((temp[['tx','tn']] < -99).any())

tx    False
tn    False
dtype: bool


In [41]:
#chossing the 30 years from which to elaborate the baseline:
baseline = temp[(temp["year"] >= 1991) & (temp["year"] <= 2020)].copy()

In [42]:
print(f"Baseline rows: {len(baseline)}")

Baseline rows: 10950


In [43]:
baseline.isna().sum()

date     0
tx       0
tn       0
year     0
month    0
day      0
doy      0
dtype: int64

In [44]:
# establishing window for calculation of average temperature of each day (-5, +5 days)
window=5

In [45]:
# variables where we will store the 90th percentile min and max temperatures for each day of the year:
tx_p90={}
tn_p90={}

In [46]:
# loop to calculate the 90thp min or max temperature for each day of the year based on a 5 day window:

for doy in range(1, 366):
    window_doys = [(doy + d - 1) % 365 + 1 for d in range(-window, window + 1)]
    subset = baseline[baseline["doy"].isin(window_doys)]
    tx_vals = subset["tx"].dropna()
    tn_vals = subset["tn"].dropna()
    tx_p90[doy] = np.percentile(tx_vals, 90)
    tn_p90[doy] = np.percentile(tn_vals, 90)
    

In [47]:
# convert to pd series for mapping
tx_p90_series = pd.Series(tx_p90)
tn_p90_series = pd.Series(tn_p90)

In [48]:
print("Sample thresholds around mid-July (DOY ~196):")
for d in range(193, 200):
    print(f"  DOY {d}: TX_p90={tx_p90[d]:.2f}°C | TN_p90={tn_p90[d]:.2f}°C")

Sample thresholds around mid-July (DOY ~196):
  DOY 193: TX_p90=28.41°C | TN_p90=16.61°C
  DOY 194: TX_p90=28.91°C | TN_p90=16.70°C
  DOY 195: TX_p90=28.91°C | TN_p90=16.81°C
  DOY 196: TX_p90=29.00°C | TN_p90=16.90°C
  DOY 197: TX_p90=29.01°C | TN_p90=16.91°C
  DOY 198: TX_p90=29.50°C | TN_p90=16.90°C
  DOY 199: TX_p90=29.60°C | TN_p90=16.91°C


# 3. APPLY THRESHOLDS TO DATASET

In [49]:
evaluation=temp.copy()

In [50]:
evaluation.head()

,date,tx,tn,year,month,day,doy
0,1979-01-01,2.3,-7.5,1979,1,1,1
1,1979-01-02,1.6,-7.5,1979,1,2,2
2,1979-01-03,1.3,-7.2,1979,1,3,3
3,1979-01-04,-0.3,-6.5,1979,1,4,4
4,1979-01-05,5.6,-1.4,1979,1,5,5


In [54]:
# create column with threshold temperatures (by doy):

evaluation["tx_p90"] = evaluation["doy"].map(tx_p90_series)
evaluation["tn_p90"] = evaluation["doy"].map(tn_p90_series)

In [55]:
# flag temperatures above the threshhold (by doy):
evaluation["above_tx_p90"] = evaluation["tx"] > evaluation["tx_p90"]
evaluation["above_tn_p90"] = evaluation["tn"] > evaluation["tn_p90"]

In [57]:
# flag days with both min and max temperatures above the threshold (by doy):
evaluation["above_baseline_day"] = (evaluation["above_tx_p90"] & evaluation["above_tn_p90"])

In [58]:
evaluation.groupby("year")["above_tx_p90"].value_counts()

year  above_tx_p90
1979  False           345
      True             20
1980  False           349
      True             16
1981  False           350
                     ... 
2023  True             58
2024  False           309
      True             56
2025  False           295
      True             70
Name: count, Length: 94, dtype: int64

In [59]:
evaluation.head()

,date,tx,tn,year,month,day,doy,TX_p90,TN_p90,tx_p90,tn_p90,above_tx_p90,above_tn_p90,above_baseline_day
0,1979-01-01,2.3,-7.5,1979,1,1,1,12.41,6.81,12.41,6.81,False,False,False
1,1979-01-02,1.6,-7.5,1979,1,2,2,12.50,6.80,12.50,6.80,False,False,False
2,1979-01-03,1.3,-7.2,1979,1,3,3,12.51,6.90,12.51,6.90,False,False,False
3,1979-01-04,-0.3,-6.5,1979,1,4,4,12.60,6.90,12.60,6.90,False,False,False
4,1979-01-05,5.6,-1.4,1979,1,5,5,12.50,7.01,12.50,7.01,False,False,False


In [60]:
# function to identify heatwaves where we create a group everytime the value in hot_col changes and sums the number of occurences. 
# second we count the number of rows in each group and store it in run_length variable
# Then we filter by those that have true in the hot col and 

def identify_heatwaves(df, hot_col, min_duration=3):
    groups = (df[hot_col] != df[hot_col].shift()).cumsum()
    run_length = df.groupby(groups)[hot_col].transform("size")
    df["heatwave"] = ((df[hot_col] == True)& (run_length >= min_duration))
    return df

In [61]:
# apply function and see result:
evaluation = identify_heatwaves(evaluation,hot_col="above_tx_p90",min_duration=3)

In [62]:
evaluation.head()

,date,tx,tn,year,month,day,doy,TX_p90,TN_p90,tx_p90,tn_p90,above_tx_p90,above_tn_p90,above_baseline_day,heatwave
0,1979-01-01,2.3,-7.5,1979,1,1,1,12.41,6.81,12.41,6.81,False,False,False,False
1,1979-01-02,1.6,-7.5,1979,1,2,2,12.50,6.80,12.50,6.80,False,False,False,False
2,1979-01-03,1.3,-7.2,1979,1,3,3,12.51,6.90,12.51,6.90,False,False,False,False
3,1979-01-04,-0.3,-6.5,1979,1,4,4,12.60,6.90,12.60,6.90,False,False,False,False
4,1979-01-05,5.6,-1.4,1979,1,5,5,12.50,7.01,12.50,7.01,False,False,False,False


In [63]:
# days in heatwave per year:
evaluation.groupby("year")["heatwave"].sum()

year
1979    10
1980     9
1981    10
1982    11
1983    16
1984    15
1985    11
1986    11
1987     7
1988     0
1989    28
1990    36
1991     6
1992    15
1993     7
1994    13
1995    33
1996     7
1997    21
1998    16
1999    13
2000     9
2001     9
2002    11
2003    29
2004    12
2005     8
2006    24
2007     9
2008    13
2009    12
2010     4
2011    26
2012    23
2013     9
2014    26
2015    34
2016    18
2017    23
2018    33
2019    27
2020    37
2021    31
2022    38
2023    33
2024    27
2025    51
Name: heatwave, dtype: int64

In [64]:
# the heatwave column gives us the days in heatwave per year NOT number of heatwaves per year, 
# so we'll create a new column counting everytime a heatwave starts so then we can sum per year and get the actuall number of heatwaves per year:

evaluation["heatwave_start"] = ((evaluation["heatwave"]==True) & (evaluation["heatwave"].astype(int).diff() > 0))
evaluation.head()

,date,tx,tn,year,month,day,doy,TX_p90,TN_p90,tx_p90,tn_p90,above_tx_p90,above_tn_p90,above_baseline_day,heatwave,heatwave_start
0,1979-01-01,2.3,-7.5,1979,1,1,1,12.41,6.81,12.41,6.81,False,False,False,False,False
1,1979-01-02,1.6,-7.5,1979,1,2,2,12.50,6.80,12.50,6.80,False,False,False,False,False
2,1979-01-03,1.3,-7.2,1979,1,3,3,12.51,6.90,12.51,6.90,False,False,False,False,False
3,1979-01-04,-0.3,-6.5,1979,1,4,4,12.60,6.90,12.60,6.90,False,False,False,False,False
4,1979-01-05,5.6,-1.4,1979,1,5,5,12.50,7.01,12.50,7.01,False,False,False,False,False


In [65]:
evaluation.groupby("year")["heatwave_start"].sum()

year
1979     2
1980     3
1981     3
1982     3
1983     4
1984     4
1985     3
1986     3
1987     2
1988     0
1989     5
1990     8
1991     2
1992     3
1993     2
1994     4
1995     7
1996     2
1997     7
1998     4
1999     3
2000     3
2001     3
2002     3
2003     8
2004     3
2005     2
2006     5
2007     3
2008     3
2009     4
2010     1
2011     5
2012     5
2013     2
2014     7
2015     7
2016     4
2017     6
2018     7
2019     5
2020     9
2021     7
2022     8
2023     5
2024     6
2025    12
Name: heatwave_start, dtype: int64

In [67]:
evaluation.groupby("year")["above_tx_p90"].sum()

year
1979    20
1980    16
1981    15
1982    22
1983    30
1984    26
1985    22
1986    16
1987    20
1988    15
1989    53
1990    52
1991    18
1992    27
1993    19
1994    28
1995    51
1996    22
1997    44
1998    33
1999    27
2000    23
2001    27
2002    30
2003    49
2004    23
2005    43
2006    48
2007    31
2008    23
2009    24
2010    19
2011    56
2012    36
2013    22
2014    46
2015    47
2016    38
2017    39
2018    59
2019    40
2020    64
2021    45
2022    64
2023    58
2024    56
2025    70
Name: above_tx_p90, dtype: int64

In [68]:
# adding the may-sep condition to see if there's a differince and if it's negligible:
evaluation["heatwave_summer"] = (evaluation["heatwave"] & evaluation["month"].between(5, 9))
evaluation.groupby("year")["heatwave_summer"].sum()

year
1979     0
1980     3
1981     3
1982    11
1983     6
1984     7
1985     3
1986     4
1987     0
1988     0
1989    25
1990    16
1991     3
1992     8
1993     4
1994     3
1995    26
1996     7
1997     8
1998     4
1999     9
2000     3
2001     9
2002     0
2003    19
2004     6
2005     4
2006    21
2007     0
2008     6
2009     6
2010     4
2011     4
2012    13
2013     9
2014     3
2015     3
2016    14
2017    12
2018    17
2019    10
2020    25
2021    18
2022    20
2023    19
2024     4
2025    23
Name: heatwave_summer, dtype: int64

In [69]:
# number of summer heatwaves filtered may-sep:
evaluation["heatwave_start_summer"] = (evaluation["heatwave_start"] & evaluation["month"].between(5, 9))
evaluation.groupby("year")["heatwave_start_summer"].sum()

year
1979    0
1980    1
1981    1
1982    3
1983    1
1984    2
1985    1
1986    1
1987    0
1988    0
1989    4
1990    3
1991    1
1992    1
1993    1
1994    1
1995    6
1996    2
1997    2
1998    1
1999    2
2000    1
2001    3
2002    0
2003    5
2004    2
2005    1
2006    4
2007    0
2008    1
2009    2
2010    1
2011    1
2012    3
2013    2
2014    1
2015    1
2016    3
2017    3
2018    4
2019    2
2020    6
2021    4
2022    4
2023    2
2024    1
2025    5
Name: heatwave_start_summer, dtype: int64

In [73]:
evaluation.to_csv(config['data']['clean']['file11'],index=False,sep=";",encoding="utf-8")